# Tine: train a small verifier on graded patches

Use a **GPU** runtime: A100, or L4 if no A100 is free. Run the cells in order. About 15 GB of model
weights are downloaded, then roughly 15-30 minutes of compute.

What it does: a frozen Qwen2.5-7B-Instruct reads each of the 861 accepted patches with its issue
once, and its hidden states are saved. A logistic regression is then trained on those states to say
whether a patch resolves its issue, and scored on tasks it never saw in training. The language model
is not trained and generates nothing. **No patch is executed.** See `docs/JUDGE.md`.

Needs three files from `artifacts/judge/` on the Mac: `cases-222.jsonl`, `key-222.csv` and
`key-pilot.csv`. Put them in `MyDrive/tine/`, or cell 1 asks you to upload them. They hold issue
text, model patches and grades, so they stay on Drive and are never committed.

In [2]:
# 1. Mount Drive, get the repository, check the GPU and the input files.
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path

BRANCH = "claude/colab-judge"  # falls back to "main" once this branch is merged and deleted
ORIGIN = "https://github.com/Nandu-Dammalapti/tine.git"
if subprocess.run(["git", "ls-remote", "--exit-code", "--heads", ORIGIN, BRANCH], capture_output=True).returncode:
    BRANCH = "main"
ROOT = Path("/content/drive/MyDrive/tine")
REPO = Path("/content/tine")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, ORIGIN, str(REPO)], check=True)
os.chdir(REPO)
print("tine revision:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU: change the runtime type")

NEEDED = ["cases-222.jsonl", "key-222.csv", "key-pilot.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    from google.colab import files
    print("Upload from artifacts/judge/ on the Mac (select all at once):", ", ".join(missing))
    uploaded = files.upload()
    ROOT.mkdir(parents=True, exist_ok=True)
    for name in missing:
        (ROOT / name).write_bytes(uploaded[name])
CASES, KEY, PILOT = (ROOT / name for name in NEEDED)
count = sum(1 for line in CASES.open(encoding="utf-8") if line.strip())
print("cases:", count)
assert count == 861, "expected the 861 cases from the 222-task run"
CACHE = "/content/hf"  # model weights stay on the runtime disk, not on Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tine revision: ac799cb11e88164bb2e97b28d2855a084ced8df7
Tesla T4, 15360 MiB
Upload from artifacts/judge/ on the Mac (select all at once): cases-222.jsonl, key-222.csv, key-pilot.csv


Saving key-pilot.csv to key-pilot (1).csv


KeyError: 'cases-222.jsonl'

In [ ]:
# 2. Smoke test on 6 cases (downloads the model: several minutes the first time).
SMOKE = Path("/content/verifier-hidden-smoke.npz")
subprocess.run([sys.executable, "scripts/train_verifier.py", "encode", "--cases", str(CASES),
                "--output", str(SMOKE), "--limit", "6", "--cache-dir", CACHE], check=True)

In [ ]:
# 3. All 861 cases. Not resumable; it is short. After a disconnect rerun cells 1 and 3.
HIDDEN = ROOT / "verifier-hidden-qwen25-7b.npz"
subprocess.run([sys.executable, "scripts/train_verifier.py", "encode", "--cases", str(CASES),
                "--output", str(HIDDEN), "--cache-dir", CACHE], check=True)

In [ ]:
# 4. Train and score on held-out tasks: 5 different splits per feature set. CPU work, a few minutes.
LAYERS = [12, 16, 21, 28]
sets = [f"hidden:{kind}_{layer}" for kind in ("last", "mean") for layer in LAYERS]
sets += ["agreement", "structure+text+agreement", "hidden:last_21+agreement", "hidden:mean_21+agreement",
         "hidden:last_28+agreement", "hidden:mean_28+agreement"]
SUMMARY = ROOT / "verifier-hidden-summary.csv"
subprocess.run([sys.executable, "scripts/train_verifier.py", "compare", "--cases", str(CASES), "--key", str(KEY),
                "--embeddings", str(HIDDEN), "--subset-key", str(PILOT), "--agreement-exclude", "claude-opus-4.1",
                "--c", "0.0001", "0.001", "0.01", "0.1", "--seeds", "5", "--output", str(SUMMARY),
                "--feature-sets", *sets], check=True, stdout=subprocess.DEVNULL)

In [ ]:
# 5. The table. Copy this output back to Claude.
import pandas as pd
table = pd.read_csv(SUMMARY)
pd.set_option("display.width", 200)
print(table[["features", "cases_scored", "cases", "auc", "auc_sd", "catch_rate", "false_reject_rate",
             "accuracy_when_accepted"]].to_string(index=False))
print(json.loads(Path(str(HIDDEN) + ".json").read_text()))
print("saved:", SUMMARY)

Done. Choose **Runtime → Disconnect and delete runtime**. The summary is on Drive at
`MyDrive/tine/verifier-hidden-summary.csv` and the hidden states at
`MyDrive/tine/verifier-hidden-qwen25-7b.npz`.

Reading the table: `auc` is how well the verifier ranks right patches above wrong ones on tasks it
never saw (0.5 is chance, 1.0 is perfect). The Claude session judge scored 0.99 on the `subset` rows
(the 150 pilot cases), the 32B open model 0.62, and patch agreement between cheap models 0.79.